## 새로운 수어 데이터 만들때 사용

In [36]:
import cv2
import os

def trim_first_and_last_half_second(video_path, output_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"영상 열기 실패: {video_path}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cut_frames = int(fps * 0.5)

    start_frame = int(cut_frames)
    end_frame = int(total_frames - cut_frames)

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if start_frame <= frame_idx < end_frame:
            out.write(frame)

        frame_idx += 1

    cap.release()
    out.release()
    print(f"저장 완료: {output_path} (앞뒤 0.5초 잘림)")

input_folder = "새로운 수어"
output_folder = "새로운 수어"

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if not filename.lower().endswith(('.mov', '.mp4', '.avi')):
        continue

    input_video = os.path.join(input_folder, filename)
    output_video = os.path.join(output_folder, filename.replace(".MOV", "_trim.mov"))

    trim_first_and_last_half_second(input_video, output_video)

print("파일 처리 완료")


✅ 저장 완료: 새로운 수어/IMG_1751_trim.mov (앞뒤 0.5초 잘림)
✅ 저장 완료: 새로운 수어/IMG_1743_trim.mov (앞뒤 0.5초 잘림)
✅ 저장 완료: 새로운 수어/IMG_1744_trim.mov (앞뒤 0.5초 잘림)
✅ 저장 완료: 새로운 수어/IMG_1756_trim.mov (앞뒤 0.5초 잘림)
✅ 저장 완료: 새로운 수어/IMG_1757_trim.mov (앞뒤 0.5초 잘림)
✅ 저장 완료: 새로운 수어/IMG_1740_trim.mov (앞뒤 0.5초 잘림)
✅ 저장 완료: 새로운 수어/IMG_1752_trim.mov (앞뒤 0.5초 잘림)
✅ 저장 완료: 새로운 수어/IMG_1749_trim.mov (앞뒤 0.5초 잘림)
✅ 저장 완료: 새로운 수어/IMG_1741_trim.mov (앞뒤 0.5초 잘림)
✅ 저장 완료: 새로운 수어/IMG_1755_trim.mov (앞뒤 0.5초 잘림)
✅ 저장 완료: 새로운 수어/IMG_1750_trim.mov (앞뒤 0.5초 잘림)
✅ 저장 완료: 새로운 수어/IMG_1746_trim.mov (앞뒤 0.5초 잘림)
✅ 저장 완료: 새로운 수어/IMG_1747_trim.mov (앞뒤 0.5초 잘림)
✅ 저장 완료: 새로운 수어/IMG_1745_trim.mov (앞뒤 0.5초 잘림)
✅ 저장 완료: 새로운 수어/IMG_1742_trim.mov (앞뒤 0.5초 잘림)
✅ 저장 완료: 새로운 수어/IMG_1748_trim.mov (앞뒤 0.5초 잘림)
✅ 저장 완료: 새로운 수어/IMG_1758_trim.mov (앞뒤 0.5초 잘림)
✅ 저장 완료: 새로운 수어/IMG_1754_trim.mov (앞뒤 0.5초 잘림)
✅ 저장 완료: 새로운 수어/IMG_1759_trim.mov (앞뒤 0.5초 잘림)
✅ 저장 완료: 새로운 수어/IMG_1753_trim.mov (앞뒤 0.5초 잘림)
🎯 모든 파일 처리 완료!


In [37]:
import os
import cv2
import numpy as np
import mediapipe as mp

base_folder = "새로운 수어"

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

def fix_frame_count(keypoints, fixed_len=30):
    cur_len = keypoints.shape[0]
    if cur_len == fixed_len:
        return keypoints
    elif cur_len > fixed_len:
        return keypoints[:fixed_len]
    else:
        pad = np.zeros((fixed_len - cur_len, *keypoints.shape[1:]))
        return np.concatenate([keypoints, pad], axis=0)

for file in os.listdir(base_folder):
    if not file.lower().endswith(".mov"):
        continue

    video_path = os.path.join(base_folder, file)
    output_filename = file.replace(".MOV", "trim_flipped_keypoints.npy").replace(".mov", "trim_flipped_keypoints.npy")
    output_path = os.path.join(base_folder, output_filename)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"영상 열기 실패: {file}")
        continue

    print(f"처리 중: {file}")
    keypoints_list = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # 좌우반전
        frame = cv2.flip(frame, 1)

        # RGB 변환
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Mediapipe 추론
        results = hands.process(frame_rgb)

        # 빈값 채우기 (2 hands)
        frame_keypoints = np.zeros((2, 21, 3))
        if results.multi_hand_landmarks:
            for i, hand_landmarks in enumerate(results.multi_hand_landmarks[:2]):
                for j, lm in enumerate(hand_landmarks.landmark):
                    frame_keypoints[i, j] = [lm.x, lm.y, lm.z]

        keypoints_list.append(frame_keypoints)

    cap.release()

    if keypoints_list:
        keypoints_array = np.array(keypoints_list)
        keypoints_array = fix_frame_count(keypoints_array, fixed_len=30)
        np.save(output_path, keypoints_array)
        print(f"저장 완료: {output_path} | shape: {keypoints_array.shape}")
    else:
        print(f"손 인식 안 됨: {file}")

hands.close()


🎬 처리 중: IMG_1751.MOV


I0000 00:00:1750409734.612882    8336 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1750409734.629147   87304 gl_context.cc:369] GL version: 3.1 (OpenGL ES 3.1 Mesa 24.2.8-1ubuntu1~24.04.1), renderer: D3D12 (NVIDIA GeForce RTX 3060 Ti)
W0000 00:00:1750409734.719192   87293 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1750409734.779409   87299 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


✅ 저장 완료: 새로운 수어/IMG_1751trim_flipped_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1747_trim.mov
✅ 저장 완료: 새로운 수어/IMG_1747_trimtrim_flipped_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1751_trim.mov
✅ 저장 완료: 새로운 수어/IMG_1751_trimtrim_flipped_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1743.MOV
✅ 저장 완료: 새로운 수어/IMG_1743trim_flipped_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1744.MOV
✅ 저장 완료: 새로운 수어/IMG_1744trim_flipped_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1742_trim.mov
✅ 저장 완료: 새로운 수어/IMG_1742_trimtrim_flipped_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1756.MOV
✅ 저장 완료: 새로운 수어/IMG_1756trim_flipped_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1756_trim.mov
✅ 저장 완료: 새로운 수어/IMG_1756_trimtrim_flipped_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1757_trim.mov
✅ 저장 완료: 새로운 수어/IMG_1757_trimtrim_flipped_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1759_trim.mov
✅ 저장 완료: 새로운 수어/IMG_1759_trimtrim_flipped_keypoints.npy | shape: (30, 2, 21, 3)
🎬

In [38]:
import os
import cv2
import numpy as np
import mediapipe as mp

base_folder = "새로운 수어"

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

def fix_frame_count(keypoints, fixed_len=30):
    cur_len = keypoints.shape[0]
    if cur_len == fixed_len:
        return keypoints
    elif cur_len > fixed_len:
        return keypoints[:fixed_len]
    else:
        pad = np.zeros((fixed_len - cur_len, *keypoints.shape[1:]))
        return np.concatenate([keypoints, pad], axis=0)

for file in os.listdir(base_folder):
    if not file.lower().endswith(".mov"):
        continue

    video_path = os.path.join(base_folder, file)
    output_filename = file.replace(".MOV", "_keypoints.npy").replace(".mov", "_keypoints.npy")
    output_path = os.path.join(base_folder, output_filename)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"영상 열기 실패: {file}")
        continue

    print(f"처리 중: {file}")
    keypoints_list = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
  
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)      
        results = hands.process(frame_rgb)
        frame_keypoints = np.zeros((2, 21, 3))
        if results.multi_hand_landmarks:
            for i, hand_landmarks in enumerate(results.multi_hand_landmarks[:2]):
                for j, lm in enumerate(hand_landmarks.landmark):
                    frame_keypoints[i, j] = [lm.x, lm.y, lm.z]

        keypoints_list.append(frame_keypoints)

    cap.release()

    if keypoints_list:
        keypoints_array = np.array(keypoints_list)
        keypoints_array = fix_frame_count(keypoints_array, fixed_len=30)
        np.save(output_path, keypoints_array)
        print(f"저장 완료: {output_path} | shape: {keypoints_array.shape}")
    else:
        print(f"손 인식 안 됨: {file}")

hands.close()


🎬 처리 중: IMG_1751.MOV


I0000 00:00:1750409881.668399    8336 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1750409881.682327   87893 gl_context.cc:369] GL version: 3.1 (OpenGL ES 3.1 Mesa 24.2.8-1ubuntu1~24.04.1), renderer: D3D12 (NVIDIA GeForce RTX 3060 Ti)
W0000 00:00:1750409881.695813   87882 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1750409881.707767   87880 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


✅ 저장 완료: 새로운 수어/IMG_1751_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1747_trim.mov
✅ 저장 완료: 새로운 수어/IMG_1747_trim_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1751_trim.mov
✅ 저장 완료: 새로운 수어/IMG_1751_trim_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1743.MOV
✅ 저장 완료: 새로운 수어/IMG_1743_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1744.MOV
✅ 저장 완료: 새로운 수어/IMG_1744_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1742_trim.mov
✅ 저장 완료: 새로운 수어/IMG_1742_trim_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1756.MOV
✅ 저장 완료: 새로운 수어/IMG_1756_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1756_trim.mov
✅ 저장 완료: 새로운 수어/IMG_1756_trim_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1757_trim.mov
✅ 저장 완료: 새로운 수어/IMG_1757_trim_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1759_trim.mov
✅ 저장 완료: 새로운 수어/IMG_1759_trim_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1753_trim.mov
✅ 저장 완료: 새로운 수어/IMG_1753_trim_keypoints.npy | shape: (30, 2, 21, 3)
🎬 처리 중: IMG_1757.MOV
✅ 저장 완

## 좌우반전 포함된 학습 코드

In [39]:
import os
import numpy as np
from sklearn.model_selection import train_test_split

base_dir = "/home/changhyoun/키워드 수어"
labels = ['가세요(2)','감기(2)','감사합니다(2)','괜찮아요(2)','기분(2)','날씨(2)', '네(2)','더워요(2)','도와드릴게요(2)','만나서반가워요(2)','밝아요(2)', '밥 먹었어요(2)','배고파요(2)',
          '버스(2)','부탁해요(2)','수고하셨습니다(2)','아니요(2)', '안녕하세요(2)','어때요,무엇,어디(2)','영화(2)','조금(2)','조심하세요(2)','졸려요(2)','좋아요(2)','지하철(2)','집(2)','추워요(2)',
         '친구(2)','학교(2)','힘들어요(2)']
sequence_length = 30
X, y = [], []

for label_idx, label in enumerate(labels):
    folder = os.path.join(base_dir, label)
    for file in os.listdir(folder):
        if file.endswith("_keypoints.npy"):
            keypoints = np.load(os.path.join(folder, file))
            if keypoints.shape == (30, 2, 21, 3):
                keypoints_flat = keypoints.reshape(sequence_length, -1)  # → (30, 126)
                X.append(keypoints_flat)
                y.append(label_idx)

X = np.array(X)
y = np.array(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [40]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(30, 126)),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(len(labels), activation='softmax')
])
early_stopping = EarlyStopping(
    monitor='val_loss',     
    patience=10,              
    restore_best_weights=True  
)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

model.fit(X_train, y_train, epochs=200, batch_size=2, validation_split=0.2,callbacks=[early_stopping])


/home/changhyoun/myenv/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 30, 128)        │       130,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 30)             │         1,950 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 186,078 (726.87 KB)

 Trainable params: 186,078 (726.87 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
1981/1981 ━━━━━━━━━━━━━━━━━━━━ 29s 14ms/step - accuracy: 0.1179 - loss: 3.0010 - val_accuracy: 0.1847 - val_loss: 2.4746
Epoch 2/200
1981/1981 ━━━━━━━━━━━━━━━━━━━━ 25s 13ms/step - accuracy: 0.1976 - loss: 2.4292 - val_accuracy: 0.3481 - val_loss: 2.0498
Epoch 3/200
1981/1981 ━━━━━━━━━━━━━━━━━━━━ 26s 13ms/step - accuracy: 0.3205 - loss: 2.0182 - val_accuracy: 0.4612 - val_loss: 1.6869
Epoch 4/200
1981/1981 ━━━━━━━━━━━━━━━━━━━━ 25s 13ms/step - accuracy: 0.4352 - loss: 1.6721 - val_accuracy: 0.5459 - val_loss: 1.3803
Epoch 5/200
1981/1981 ━━━━━━━━━━━━━━━━━━━━ 25s 13ms/step - accuracy: 0.5540 - loss: 1.3477 - val_accuracy: 0.5520 - val_loss: 1.3941
Epoch 6/200
1981/1981 ━━━━━━━━━━━━━━━━━━━━ 25s 13ms/step - accuracy: 0.6198 - loss: 1.1633 - val_accuracy: 0.7104 - val_loss: 0.9389
Epoch 7/200
1981/1981 ━━━━━━━━━━━━━━━━━━━━ 25s 13ms/step - accuracy: 0.6984 - loss: 0.9566 - val_accuracy: 0.7588 - val_loss: 0.8422
Epoch 8/200
1981/1981 ━━━━━━━━━━━━━━━━━━━━ 25s 13ms/step - accuracy: 

In [41]:
model.save("6/201.h5")

In [52]:
import numpy as np
import cv2
file = "./키워드 수어/아니요(2)/IMG_9563_fixed_keypoints.npy"
sample = np.load(file)

if sample.shape == (30, 2, 21, 3):
    sample = sample.reshape(1, 30, 126)
    pred = model.predict(sample)
    pred_label = labels[np.argmax(pred)]
    print("예측 결과:", pred_label)
else:
    print("입력 shape가 올바르지 않습니다.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step
✅ 예측 결과: 아니요(2)


In [94]:
## import cv2
import numpy as np
import mediapipe as mp
from collections import deque
from tensorflow.keras.models import load_model
import cv2
labels = ['가세요(2)','감기(2)','감사합니다(2)','괜찮아요(2)','기분(2)','날씨(2)', '네(2)','더워요(2)','도와드릴게요(2)','만나서반가워요(2)','밝아요(2)', '밥 먹었어요(2)','배고파요(2)',
          '버스(2)','부탁해요(2)','수고하셨습니다(2)','아니요(2)', '안녕하세요(2)','어때요(2)','영화(2)','조금(2)','조심하세요(2)','졸려요(2)','좋아요(2)','지하철(2)','집(2)','추워요(2)',
         '친구(2)','학교(2)','힘들어요(2)']

model = load_model("6/201.h5")

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

video_path = "테스트합시다/힘들어요1.MOV"
cap = cv2.VideoCapture(video_path)

frame_buffer = deque(maxlen=30)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break


    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    
    keypoints = np.zeros((2, 21, 3))  # (2 hands, 21 landmarks, 3 coords)
    if result.multi_hand_landmarks:
        for i, hand_landmarks in enumerate(result.multi_hand_landmarks[:2]):
            for j, lm in enumerate(hand_landmarks.landmark):
                keypoints[i, j] = [lm.x, lm.y, lm.z]

 
    frame_buffer.append(keypoints)

   
    if len(frame_buffer) == 30:
        input_data = np.array(frame_buffer).reshape(1, 30, 126)
        pred = model.predict(input_data)[0]
        pred_label = labels[np.argmax(pred)]
        confidence = np.max(pred)

        print(f"👉 슬라이딩 예측: {pred_label} (신뢰도: {confidence:.2f})")

cap.release()
hands.close()


I0000 00:00:1750412425.370197    8336 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1750412425.388687  258226 gl_context.cc:369] GL version: 3.1 (OpenGL ES 3.1 Mesa 24.2.8-1ubuntu1~24.04.1), renderer: D3D12 (NVIDIA GeForce RTX 3060 Ti)
W0000 00:00:1750412425.400716  258214 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1750412425.412869  258220 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step
👉 슬라이딩 예측: 힘들어요(2) (신뢰도: 0.90)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
👉 슬라이딩 예측: 힘들어요(2) (신뢰도: 0.90)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
👉 슬라이딩 예측: 힘들어요(2) (신뢰도: 0.90)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
👉 슬라이딩 예측: 힘들어요(2) (신뢰도: 0.90)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
👉 슬라이딩 예측: 힘들어요(2) (신뢰도: 0.90)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
👉 슬라이딩 예측: 힘들어요(2) (신뢰도: 0.90)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
👉 슬라이딩 예측: 힘들어요(2) (신뢰도: 0.90)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
👉 슬라이딩 예측: 힘들어요(2) (신뢰도: 0.90)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
👉 슬라이딩 예측: 힘들어요(2) (신뢰도: 0.90)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
👉 슬라이딩 예측: 힘들어요(2) (신뢰도: 0.90)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
👉 슬라이딩 예측: 힘들어요(2) (신뢰도: 0.90)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
👉 슬라이딩 예측: 힘들어요(2) (신뢰도: 0.90)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
👉 슬라이딩 예측: 힘들어요(2) (신뢰도: 0.90)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
👉 슬라이딩 예측: 힘들어요(2) (신뢰도: 0.90)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/

### 데이터 다시 로딩

In [ ]:
import os
import numpy as np
from sklearn.model_selection import train_test_split

base_dir = "/home/changhyoun/키워드 수어"
labels = ['감사합니다(2)', '네(2)', '아니요(2)', '안녕하세요(2)']
sequence_length = 30
X, y = [], []

for label_idx, label in enumerate(labels):
    folder = os.path.join(base_dir, label)
    for file in os.listdir(folder):
        if file.endswith("_keypoints.npy"):
            keypoints = np.load(os.path.join(folder, file))
            if keypoints.shape == (30, 2, 21, 3):
                X.append(keypoints.reshape(30, -1))  # (30, 126)
                y.append(label_idx)

X = np.array(X)
y = np.array(y)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


### 기존 모델 로드 & 이어서 학습

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping


model = load_model("sign_model_v1.h5")


early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)


model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=8,
    callbacks=[early_stop]
)


model.save("sign_model_v2_finetuned.h5")
